# 第6章　勾配ブースティング

## 6.4　Pythonで勾配ブースティング

### コード6.1　データの取得

In [ ]:
!pip install japanize-matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.datasets import fetch_california_housing
import japanize_matplotlib

# データセットをロード
print("California Housing データセットをロードしています...")
data = fetch_california_housing(as_frame=True)
df = data.frame

# データセットの説明を保存
data_description = data.DESCR
with open("dataset_description.txt", "w") as f:
    f.write(data_description)

df

### コード6.2　データの理解

In [ ]:
# ヒストグラムを表示して保存
df.hist(figsize=(15, 15))
plt.tight_layout()
histogram_path = "histogram.svg"
plt.savefig(histogram_path, format='svg', dpi=300)
plt.show()

# ペアプロットを作成し保存
sns.pairplot(df, diag_kind="hist", plot_kws={'color': 'black'})
plt.savefig("pairplot.svg", format='svg', dpi=300)
plt.show()

### コード6.3　データの分割，回帰木モデルの学習

In [ ]:
# 特徴量 (X) とターゲット (y) に分割
X = df.drop("MedHouseVal", axis=1)
# ターゲット (住宅価格)
y = df["MedHouseVal"]

# データをトレイン (トレーニング用) とテスト用に分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Gradient Boosting Regressor のモデルを定義
# ハイパーパラメータを指定 (初学者向けに設定)
gbm = GradientBoostingRegressor(
    n_estimators=100,       # 木の数 (弱学習器の数)
    learning_rate=0.1,      # 学習率
    max_depth=3,            # 各木の深さ
    min_samples_split=2,    # ノード分割の最小サンプル数
    subsample=0.8,          # サブサンプル比率
    random_state=42         # 再現性のためのランダムシード
)

# モデルの学習
print("Gradient Boosting Regressor をトレーニングしています...")
gbm.fit(X_train, y_train)

### コード6.4　回帰木モデルの評価

In [ ]:
# イテレーションごとのMSE (平均二乗誤差) を記録
train_mse = []
test_mse = []

for y_pred_train, y_pred_test in zip(gbm.staged_predict(X_train), gbm.staged_predict(X_test)):
    train_mse.append(mean_squared_error(y_train, y_pred_train))
    test_mse.append(mean_squared_error(y_test, y_pred_test))

# イテレーションごとのMSEをプロット
plt.figure(figsize=(10, 6))
plt.plot(train_mse, label='トレーニングデータ MSE', color='black', linestyle='--')
plt.plot(test_mse, label='テストデータ MSE', color='gray')
plt.xlabel("イテレーション")
plt.ylabel("MSE")
plt.legend()
plt.title("イテレーションごとのMSEの変化")
plt.savefig("mse_vs_iteration.svg", format='svg', dpi=300)
plt.show()

# 特徴量の重要度を可視化
print("特徴量の重要度をプロットしています...")
feature_importances = gbm.feature_importances_
feature_names = X.columns

plt.figure(figsize=(10, 6))
plt.barh(feature_names, feature_importances, color='black')
plt.xlabel("重要度")
plt.ylabel("特徴量")
plt.title("特徴量の重要度")
plt.tight_layout()
plt.savefig("feature_importances.svg", format='svg', dpi=300)
plt.show()

# 最終的なMSEを計算
print("最終評価を計算しています...")
final_train_mse = train_mse[-1]
final_test_mse = test_mse[-1]

# 評価指標を保存
metrics = {
    "最終トレーニング MSE": final_train_mse,
    "最終テスト MSE": final_test_mse
}
pd.DataFrame([metrics]).to_csv("final_metrics.csv")

print("すべての結果が保存されました")